In [ ]:
# ---------------------------------------------------------------------
# 0) Load libraries & data  (UNCHANGED from your original script) ------
# ---------------------------------------------------------------------
suppressPackageStartupMessages({
  library(cmapR); library(data.table); library(tidyverse)
  library(limma); library(edgeR)
})

compound_gctx <- "/home/ubuntu/data/level3_beta_trt_cp.gctx"
control_gctx  <- "/home/ubuntu/data/level3_beta_ctl.gctx"
out_dir       <- "/home/ubuntu/output/per_cell_line"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

message("Parsing GCTX …")
gct_cp  <- parse_gctx(compound_gctx)
gct_ctl <- parse_gctx(control_gctx)

expr_mat  <- cbind(gct_cp@mat, gct_ctl@mat)  # genes × all samples
feat_meta <- gct_cp@rdesc                    # row meta
cdesc     <- bind_rows(
  as_tibble(gct_cp@cdesc) %>% mutate(treat = "drug"),
  as_tibble(gct_ctl@cdesc) %>% mutate(treat = "ctl")
)
rm(gct_cp, gct_ctl); gc()

rownames(expr_mat) <- feat_meta$id  # set once, globally

# ---------------------------------------------------------------------
# 1) Sample‑level metadata  -------------------------------------------
# ---------------------------------------------------------------------
meta     <- fread("/home/ubuntu/data/instinfo_beta.txt", sep = "\t", showProgress = FALSE)

sample_meta <- cdesc %>%
  rename(sample_id = id) %>%
  left_join(meta, by = "sample_id") %>%        # brings in pert_dose, pert_time, etc.
  mutate(
    cell_line = factor(cell_mfc_name),

    # ---- Standardise TIME ------------------------------------------
    time_val  = coalesce(as.numeric(pert_time),           # numeric column
                         as.numeric(str_extract(pert_itime, "^[0-9\\.]+"))),
    time_unit = coalesce(pert_time_unit,
                         str_extract(pert_itime, "[a-zA-Z]+$")),
    time_unit = recode(time_unit,
                       `h`="h", `hr`="h", `hrs`="h", `hour`="h", `hours`="h",
                       `d`="d", `day`="d", `days`="d",
                       `m`="m", `min`="m", `mins`="m", .default = NA_character_),
    time_h    = case_when(
      time_unit == "h"                ~ time_val,
      time_unit == "d"                ~ time_val*24,
      time_unit == "m"                ~ time_val/60,
      TRUE                            ~ NA_real_),
    time_bin  = ifelse(is.na(time_h), NA_character_,
                       paste0(formatC(time_h, format="fg", digits=3), "h")),

    # ---- Standardise DOSE ------------------------------------------
    dose_val  = suppressWarnings(as.numeric(pert_dose)),
    dose_unit = pert_dose_unit,
    dose_bin  = case_when(
      dose_unit %in% c("uM","µM")            ~ paste0(dose_val, "_uM"),
      dose_unit %in% c("ng","ng/ml","ng/uL") ~ paste0(dose_val, "_ng"),
      dose_unit %in% c("%")                  ~ paste0(dose_val, "_pct"),
      TRUE                                   ~ NA_character_)
  ) %>%
  filter(!is.na(time_bin))              %>%   # drop samples with unknown time
  mutate(treat     = factor(treat, levels = c("ctl","drug")),
         dose_bin  = factor(dose_bin),
         time_bin  = factor(time_bin))

rm(meta, cdesc); gc()

# ---------------------------------------------------------------------
# 2) Landmark gene index (UNCHANGED) ----------------------------------
# ---------------------------------------------------------------------
landmark_ids <- fread("/home/ubuntu/data/landmark_list.tsv") %>%
  filter(pr_is_lm == 1) %>% pull(pr_gene_id) %>% as.character()
lm_idx <- feat_meta$id %in% landmark_ids
stopifnot(sum(lm_idx) >= 2)

# ---------------------------------------------------------------------
# 3) Build a table of groups to test  ---------------------------------
# ---------------------------------------------------------------------
run_tbl <- sample_meta %>%
  filter(treat == "drug") %>%                    # each *perturbed* combo defines a group
  select(cell_line, pert_id, dose_bin, time_bin) %>%
  distinct() %>%
  arrange(cell_line, pert_id, dose_bin, time_bin)

message("Total groups to attempt: ", nrow(run_tbl))

# Convenience helper that returns TRUE if ≥2 ctl & ≥2 drug samples exist
has_reps <- function(meta_sub) {
  tbl <- table(meta_sub$treat)
  all(c("ctl","drug") %in% names(tbl)) && all(tbl[c("ctl","drug")] >= 2L)
}

# ---------------------------------------------------------------------
# 4) Loop over groups, run Limma, write to disk -----------------------
# ---------------------------------------------------------------------
for (ii in seq_len(nrow(run_tbl))) {
  row     <- run_tbl[ii, ]
  cl      <- row$cell_line
  pid     <- row$pert_id
  d_bin   <- row$dose_bin
  t_bin   <- row$time_bin

  lm_out  <- file.path(out_dir,
                       glue::glue("DEA_landmark_{cl}_{pid}_{d_bin}_{t_bin}.csv"))
  full_out<- file.path(out_dir,
                       glue::glue("DEA_full_{cl}_{pid}_{d_bin}_{t_bin}.csv"))
  if (file.exists(lm_out) && file.exists(full_out)) next

  message(">> ", cl, " | ", pid, " | ", d_bin, " | ", t_bin)

  sel_idx <- which(                             # drug WITH specific dose & time
    (sample_meta$cell_line == cl &
     sample_meta$pert_id  == pid &
     sample_meta$dose_bin == d_bin &
     sample_meta$time_bin == t_bin &
     sample_meta$treat    == "drug")            |

    # matching controls: same cell line & time_bin
    (sample_meta$cell_line == cl &
     sample_meta$treat    == "ctl" &
     sample_meta$time_bin == t_bin)
  )

  if (length(sel_idx) < 4L) {                   # at least 2+2 required
    message("   … skipped (fewer than 4 total samples)")
    next
  }
  meta_sub <- sample_meta[sel_idx, ]
  if (!has_reps(meta_sub)) {
    message("   … skipped (not enough replicates)")
    next
  }

  ex_sub  <- expr_mat[, sel_idx, drop = FALSE]
  design  <- model.matrix(~ 0 + treat, data = meta_sub)
  colnames(design) <- c("ctl","drug")

  ## ---- Landmark‑only ------------------------------------------------
  if (!file.exists(lm_out)) {
    fit_lm   <- lmFit(ex_sub[lm_idx, , drop = FALSE], design)
    fit_lm2  <- eBayes(contrasts.fit(fit_lm, makeContrasts(drug-ctl, levels = design)))
    tbl_lm   <- topTable(fit_lm2, number = Inf, sort.by = "P") %>%
                  rownames_to_column("gene") %>%
                  mutate(cell_line = cl, pert_id = pid,
                         dose_bin = d_bin, time_bin = t_bin)
    fwrite(tbl_lm, lm_out)
    rm(fit_lm, fit_lm2, tbl_lm); gc()
  }

  ## ---- Full‑gene ----------------------------------------------------
  if (!file.exists(full_out)) {
    fit_full  <- lmFit(ex_sub, design)
    fit_full2 <- eBayes(contrasts.fit(fit_full, makeContrasts(drug-ctl, levels = design)))
    tbl_full  <- topTable(fit_full2, number = Inf, sort.by = "P") %>%
                   rownames_to_column("gene") %>%
                   mutate(cell_line = cl, pert_id = pid,
                          dose_bin = d_bin, time_bin = t_bin)
    fwrite(tbl_full, full_out)
    rm(fit_full, fit_full2, tbl_full); gc()
  }

  rm(ex_sub, meta_sub, design); gc()
}


Parsing GCTX …

reading /home/ubuntu/data/level3_beta_trt_cp.gctx

done

reading /home/ubuntu/data/level3_beta_ctl.gctx

